# Multi-Tool Orchestration

A single tool is rarely enough for real-world tasks. A research question may require a web search, a Wikipedia lookup, a calculation, and a code execution step — all in one agent run. The challenge is not giving the LLM the tools; it is teaching it when to use each one and how to combine their outputs.

**What you'll build:** an agent with six tools covering search, knowledge lookup, calculation, code execution, weather, and news — with a clear system prompt that defines each tool's role.

**Time:** ~25 min | **Difficulty:** Intermediate

**What you'll learn:**
- How to assemble a diverse toolset and describe each tool's purpose clearly
- Why `description` quality determines tool selection accuracy
- How `FunctionCallingAgent` handles parallel tool calls in a single LLM response
- How to inspect which tools were used and in what order
- Patterns for grouping related tools into logical sets

## Prerequisites

You need:
- An **OpenAI API key** (`OPENAI_API_KEY`)
- The `synapsekit` package (installed below)

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key here
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: Import a diverse tool set

Import seven tools from different domains: search, knowledge, academic, news, weather, math, and code execution.

In [ ]:
import asyncio
from synapsekit.agents import (
    ArxivSearchTool,
    CalculatorTool,
    CodeInterpreterTool,
    DuckDuckGoSearchTool,
    FunctionCallingAgent,
    NewsTool,
    WeatherTool,
    WikipediaTool,
    ActionEvent,
    FinalAnswerEvent,
    ObservationEvent,
)
from synapsekit.llms.openai import OpenAILLM

## Step 2: Configure tools with clear purposes

The description field is the only signal the LLM uses to decide which tool to call. Vague descriptions cause the agent to pick the wrong tool. Precise, exclusive descriptions minimize overlap.

In [ ]:
# Each tool description must answer: "Use this tool when the user asks about X"
tools = [
    DuckDuckGoSearchTool(),   # current events, how-to questions, general web facts
    WikipediaTool(),           # definitions, historical facts, stable reference knowledge
    ArxivSearchTool(),         # academic papers, research findings, scientific studies
    NewsTool(),                # news headlines, press releases, current events summaries
    WeatherTool(),             # current weather conditions and forecasts
    CalculatorTool(),          # arithmetic, unit math, percentages, numerical calculations
    CodeInterpreterTool(timeout=10.0),  # data generation, algorithmic computation, data analysis
]

print(f"Configured {len(tools)} tools:")
for t in tools:
    print(f"  {t.name}: {t.description[:60]}...")

## Step 3: Write a system prompt that defines boundaries

A good system prompt for a multi-tool agent has three parts: the agent's role, a mapping of when to use each tool, and explicit fallback rules.

In [ ]:
SYSTEM_PROMPT = """
You are a versatile research and analysis assistant with access to seven tools.

Tool selection guide:
- duck_duck_go_search: Use for current events, product info, or any web fact
- wikipedia: Use for definitions, biographies, history, and stable reference facts
- arxiv_search: Use for academic papers and scientific research
- news: Use for news headlines and recent announcements
- weather: Use ONLY for weather-related questions; requires a city name
- calculator: Use for any arithmetic, percentages, or unit conversions
- code_interpreter: Use for generating data, running algorithms, or data analysis

Rules:
1. When a question requires multiple steps, plan all tool calls before executing.
2. Prefer the most specific tool over a general web search.
3. Always cite sources when using search results.
4. If a tool returns an error, report it and try an alternative approach.
"""

agent = FunctionCallingAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    max_iterations=12,
)

print("Agent configured with 7 tools and boundary-setting system prompt.")

## Step 4: Handle parallel tool calls

When `FunctionCallingAgent` receives a response with multiple `tool_calls` in one LLM message, it executes them all before returning observations to the LLM. This halves the number of round-trips for questions that require independent parallel lookups.

In [ ]:
async def run_with_trace(question: str) -> None:
    print(f"Q: {question}\n")
    tools_used = []

    async for event in agent.stream_steps(question):
        if isinstance(event, ActionEvent):
            tools_used.append(event.tool)
            print(f"  [{event.tool}] {str(event.tool_input)[:80]}")
        elif isinstance(event, ObservationEvent):
            print(f"  -> {event.observation[:120]}...")
        elif isinstance(event, FinalAnswerEvent):
            print(f"\nAnswer: {event.answer}")
            print(f"\nTools used: {tools_used}")

asyncio.run(run_with_trace("What is 15% of 847 + 23% of 1200?"))

## Step 5: Inspect tool usage patterns

After a run, `agent.memory.steps` tells you the order and frequency of tool calls. This is useful for identifying when the agent over-searches or picks the wrong tool.

In [ ]:
def summarize_run(agent: FunctionCallingAgent) -> None:
    from collections import Counter
    tool_counts = Counter(step.action for step in agent.memory.steps)
    print("\nTool usage:")
    for tool_name, count in tool_counts.most_common():
        print(f"  {tool_name}: {count}x")
    print(f"Total steps: {len(agent.memory.steps)}")

summarize_run(agent)

## Step 6: Handle complex multi-part questions

Some questions naturally decompose into parallel sub-questions. Guide the agent by phrasing questions with explicit structure.

In [ ]:
multi_part_questions = [
    # This triggers parallel calls: weather + news simultaneously
    "What is the current weather in Tokyo, and what are today's top tech news headlines?",
    # This triggers sequential calls: wikipedia → calculator
    "What is the population of Brazil? Calculate what percentage it is of the world population of 8.1 billion.",
    # This triggers a code + search chain
    "Generate the first 20 Fibonacci numbers and explain their connection to the golden ratio (search for context).",
]

# Run just the first one to demonstrate
agent.memory.clear()
asyncio.run(run_with_trace(multi_part_questions[1]))

## Complete Working Example

A self-contained script that builds a 7-tool agent and answers three questions that trigger different tool combinations, printing the tool call trace and usage summary for each.

In [ ]:
import asyncio
from collections import Counter
from synapsekit.agents import (
    ActionEvent,
    ArxivSearchTool,
    CalculatorTool,
    CodeInterpreterTool,
    DuckDuckGoSearchTool,
    FinalAnswerEvent,
    FunctionCallingAgent,
    NewsTool,
    ObservationEvent,
    WeatherTool,
    WikipediaTool,
)
from synapsekit.llms.openai import OpenAILLM

SYSTEM_PROMPT = """
You are a versatile assistant with seven specialized tools.
Always use the most specific tool for each part of a question.
For multi-part questions, plan all tool calls upfront and execute them efficiently.
Cite sources when using search or news results.
"""


def build_agent() -> FunctionCallingAgent:
    return FunctionCallingAgent(
        llm=OpenAILLM(model="gpt-4o-mini"),
        tools=[
            DuckDuckGoSearchTool(),
            WikipediaTool(),
            ArxivSearchTool(),
            NewsTool(),
            WeatherTool(),
            CalculatorTool(),
            CodeInterpreterTool(timeout=10.0),
        ],
        system_prompt=SYSTEM_PROMPT,
        max_iterations=12,
    )


async def run_question(agent: FunctionCallingAgent, question: str) -> None:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print("=" * 60)
    tools_used = []

    async for event in agent.stream_steps(question):
        if isinstance(event, ActionEvent):
            tools_used.append(event.tool)
            print(f"  [{event.tool}]  {str(event.tool_input)[:90]}")
        elif isinstance(event, ObservationEvent):
            print(f"    {event.observation[:150]}...")
        elif isinstance(event, FinalAnswerEvent):
            print(f"\nAnswer:\n{event.answer}")

    counts = Counter(tools_used)
    print(f"\nTools used: {dict(counts)}")


async def main() -> None:
    agent = build_agent()

    questions = [
        "What is 15% of 847 + 23% of 1200?",
        "Summarize what Wikipedia says about the Fibonacci sequence, then generate the first 15 numbers in code.",
        "What is the current weather in London, and are there any related climate news stories today?",
    ]

    for question in questions:
        await run_question(agent, question)


asyncio.run(main())

## Next Steps

- [Agent with Safety Guardrails](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/agent-with-guardrails) — add input/output validation before running a multi-tool agent in production
- [Streaming Agent Responses](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/streaming-agent) — display each tool call in real time as it happens
- [ReAct Research Assistant](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/react-research-assistant) — use `ReActAgent` when your LLM does not support native function calling